# <span style="color: blue"> Week 3 HW by Minsu Sohn (Student Id: 003685904)</span>

# Exploring Class Demo Code

# 1. Softmax with Temperature

In [5]:
import sys
import tensorflow as tf
import numpy as np
import warnings
warnings.filterwarnings('ignore')

logits = np.array(
    [2.95666, 2.5522, 0.93532, 0.222642, -0.43565, -0.17562, -2.02529, -2.12347, -0.64801, -0.68092]
)

In [6]:
#Using exp to make all positive numbers
exp_logits = np.exp(logits)
exp_logits

array([19.23362405, 12.83531043,  2.5480287 ,  1.24937322,  0.64684408,
        0.83893672,  0.13195557,  0.11961584,  0.52308568,  0.50615112])

In [7]:
#Normalize the exp logits, softmax
softmax_logits = exp_logits/np.sum(exp_logits, axis=0)
softmax_logits

array([0.49785575, 0.3322376 , 0.06595485, 0.0323396 , 0.01674334,
       0.02171559, 0.00341562, 0.00309621, 0.01353989, 0.01310155])

In [8]:
#convert to tensor and later use the TensorFlow softmax function
tf_logits = tf.convert_to_tensor(logits)
tf_logits = tf.expand_dims(tf_logits, axis=0)
print(tf_logits.shape)
print(tf_logits)

(1, 10)
tf.Tensor(
[[ 2.95666   2.5522    0.93532   0.222642 -0.43565  -0.17562  -2.02529
  -2.12347  -0.64801  -0.68092 ]], shape=(1, 10), dtype=float64)


In [9]:
# Add temperature t (0 <= t <=2)
# Which word index to sample? Return the index based on the probabiity

def index_with_t(logits, t=1.0):
    if t == 0.0:
        the_index = tf.argmax(logits, axis=-1) #divided by 0 is an error, so have this condition
    else:
        adjusted_logits = logits / t
        probabilities = tf.nn.softmax(adjusted_logits)
        tf.print("Probabilities:", probabilities)

        # tf.random.categorical is used to draw samples from a categorical distribution defined by the provided logits. 
        # For example, the first logits turns to be 50%, then 50% of the samples are the first index
        samples = tf.random.categorical(logits=adjusted_logits, num_samples=100)
        tf.print("Samples for 100 times:", samples)

        # Count occurrences of 0 and 1 using TensorFlow operations, as two examples
        count0 = tf.reduce_sum(tf.cast(tf.equal(samples, 0), tf.int32))
        count1 = tf.reduce_sum(tf.cast(tf.equal(samples, 1), tf.int32))
        tf.print(f"count 0: {count0} count 1: {count1}")

        the_index = tf.random.categorical(logits=adjusted_logits, num_samples=1) # num_samples=1, just return one sample base 
                                                                         #on the probability after applying the temperature.

    return tf.squeeze(the_index)  # Squeeze to remove any singleton dimensions

In [10]:
#use original logits->tf_logits to call
temperature = 0.0
index_with_t(tf_logits, temperature)

<tf.Tensor: shape=(), dtype=int64, numpy=0>

In [11]:
temperature = 1.0
index_with_t(tf_logits, temperature)

Probabilities: [[0.49785574995629556 0.33223760024601295 0.06595484731714038 ... 0.0030962149336864835 0.013539893154372913 0.013101547805814207]]
Samples for 100 times: [[0 0 0 ... 1 0 0]]
count 0: 60 count 1: 26


<tf.Tensor: shape=(), dtype=int64, numpy=1>

In [12]:
temperature = 0.2
index_with_t(tf_logits, temperature)

Probabilities: [[0.88308572182725353 0.11687702061034259 3.6034526744994594e-05 ... 8.2156273341772864e-12 1.3138972768250317e-08 1.1145466620051779e-08]]
Samples for 100 times: [[0 0 1 ... 0 0 0]]
count 0: 90 count 1: 10


<tf.Tensor: shape=(), dtype=int64, numpy=0>

In [13]:
temperature = 2.0
index_with_t(tf_logits, temperature)

Probabilities: [[0.3014942360117282 0.2462927571999039 0.10973637497982014 ... 0.023776219447392628 0.049720429166807724 0.048908974061273126]]
Samples for 100 times: [[3 8 1 ... 8 1 5]]
count 0: 27 count 1: 19


<tf.Tensor: shape=(), dtype=int64, numpy=1>

# 2. Model Settings and Best Practices of Prompt

## 2.1 Model Settings

In [14]:
import json

import os
from dotenv import load_dotenv

import openai
from openai import OpenAI

In [15]:
load_dotenv()

True

In [16]:
client = OpenAI()

### 2.1.1 Using temperature in GPT

In [17]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system","content": "you are a nice robot"},
        {"role": "user", "content": "What is Silcon Valley famous for"},
    
    ],
    temperature=1.0,
)

In [18]:
response.choices[0].message.content

"Silicon Valley is famous for being a global center for technology and innovation. Located in the southern part of the San Francisco Bay Area in California, it is home to many of the world's largest and most influential tech companies, including Google, Apple, Facebook (now Meta), and Tesla. Key aspects that contribute to its fame include:\n\n1. **Tech Startups**: Silicon Valley is known for its vibrant startup ecosystem, where entrepreneurs create innovative products and services, often in cutting-edge fields like software, hardware, biotech, and artificial intelligence.\n\n2. **Venture Capital**: The region attracts significant venture capital investment, providing startups with the funding needed to grow and develop new technologies.\n\n3. **Research and Development**: Many major universities and research institutions in the area, such as Stanford University and UC Berkeley, contribute to technological advancements and provide a talent pool for tech companies.\n\n4. **Culture of Inn

### 2.1.2 Define helper functions
We will create helper functions will make the api call easier. 

In [19]:
def chat_complete_prompt(prompt):
    # query against the model "gpt-3.5-turbo-0613"
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        #model="gpt-4-1106-preview",
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

In [20]:
def chat_complete_messages(messages, temperature=0):
    # query against the model "gpt-3.5-turbo-1106"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages= messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

### 2.1.3 Techniques of using prompts

## Explanation: I modified the text with our project AI refrigerator. 

**Example 1: using delimiters --- """ ```**

In [21]:
text = f"""I have a refrigerator in my house. 5 house mates share the refrigerator. Each housemate keeps their grocery items with 
their name labled. And no one uses any others item. The refigerator has AI function. It knows exactly what and how many items
are currently been kept in the refrigerator. Also it knows when the food is inserted and when it is expired, 
the AI refrigerator can remind a target user when the grocery item is expired. Also the AI refrigerator generates a wonderful 
recipe with the grocery items of the owner in the refrigerator. The recipe has to be generated only with the items owned by the targer user, 
not other users items. By summarizing, the AI refrigerator has 3 functionality, food management for each user, expiration reminding and 
generating a wonderful recipe. 
"""
prompt = f"""
Summarize the text delimited by triple dashes
into a single sentence less than 25 words.
---{text}---
"""
response = chat_complete_prompt(prompt)
print(response)
print("Length of the sentence: ", len(response.split()))

The AI refrigerator manages food for each housemate, reminds them of expirations, and generates recipes using only their own grocery items.
Length of the sentence:  21


**Example 2: using structured output**

In [22]:
prompt = f"""
Please provide one random recipe, along with grocery item owner, grocery item name and grocery item count. The response 
must be a valid JSON object without any markdown formatting or extra text, and it should contain the following keys: housemate_name, ite_name and item_quanity.
"""



response = chat_complete_prompt(prompt)

In [23]:
response

'{\n  "housemate_name": "Alex",\n  "item_name": "Chicken Breast",\n  "item_quantity": 2\n}'

In [24]:
try:
    json_response = json.loads(response)
    print(json_response)  # Valid JSON object
except json.JSONDecodeError as e:
    print(f"Error parsing JSON: {e}")

{'housemate_name': 'Alex', 'item_name': 'Chicken Breast', 'item_quantity': 2}


**Example 3: if conditions are met**

In [25]:
text = f"""
When approaching a fitness goal, you have multiple strategies: working out independently, hiring a personal trainer, 
or joining a fitness group. Exercising on your own offers flexibility and a sense of self-motivation 
but may require extra discipline. Hiring a personal trainer provides expert guidance and faster results 
but often involves added expenses. Participating in a fitness group builds a sense of community and 
motivation but requires coordination and teamwork. Select the approach that best fits your schedule, budget, 
and desired level of support.
"""
prompt = f"""
A paragraph with text is delimited by triple quotes. If it contains different options, please re-write in the
following format:
Option 1: - ...
Option 2: - ...
...
Option N: -...
If the paragraph doe not contain different options, simply write "No options provided."
\"\"\"{text}\"\"\"
"""
response = chat_complete_prompt(prompt)
print("Completion of the text:")
print(response)

Completion of the text:
Option 1: - Working out independently offers flexibility and a sense of self-motivation but may require extra discipline.  
Option 2: - Hiring a personal trainer provides expert guidance and faster results but often involves added expenses.  
Option 3: - Joining a fitness group builds a sense of community and motivation but requires coordination and teamwork.  


**Example 4: few shots prompting**

In [26]:
text = f"""
I think it is a reasonable suggestion to proceed. \n
"""
prompt = f"""
Complete the response in angle brackets.
Circulation revenue has increased by 5% in Finland. \n Positive
Panostaja did not disclose the purchase price. \n Neutral
Paying off the national debt will be extremely painful. \n Negative
The acquisition will have an immediate positive impact. \n Positive
<{text}>
"""
response = chat_complete_prompt(prompt)
print("Completion of the text: ")
print(response)

Completion of the text: 
< Neutral >


**Example 5: provide instruction steps**

In [27]:
text = f"""I have a refrigerator in my house. 5 house mates share the refrigerator. Each housemate keeps their grocery items with 
their name labled. And no one uses any others item. The refigerator has AI function. It knows exactly what and how many items
are currently been kept in the refrigerator. Also it knows when the food is inserted and when it is expired, 
the AI refrigerator can remind a target user when the grocery item is expired. Also the AI refrigerator generates a wonderful 
recipe with the grocery items of the owner in the refrigerator. The recipe has to be generated only with the items owned by the targer user, 
not other users items. By summarizing, the AI refrigerator has 3 functionality, food management for each user, expiration reminding and 
generating a wonderful recipe. 

Students can only arrive in the US within thirty (30) days of the program starting date and must report
to school within seven (7) days after arriving in the U.S. or the I-20 will be voided. A student wishing
to transfer at the end of a semester must apply during that semester to allow enough processing time with the
U.S. Citizenship and Immigration Services (USCIS).

House mates must log in into the refrigeator for checking in and out the grocery items.
When checking in, the quatity of the item must be provided as well. 
House mates can not check out other house mates items. 
AI regrigerator cant use other house mates items for generating recipe. 
"""
prompt = f"""
The text will be in triple backticks
Step 1: Please summarize the description.
Step 2: Translate the step 1 summary into Chinese
Step 3: Output a valid JSON object without any markdown formatting or extra text, with both the step1 summary and step2 Chinese translation. 
'''{text}'''
"""

In [28]:
response = chat_complete_prompt(prompt)
print("Completion of the text: ")
print(response)

Completion of the text: 
{
  "summary": "The AI refrigerator is shared by five housemates, each labeling their groceries. It tracks items, expiration dates, and generates recipes using only the owner's items. Housemates must log in to manage their groceries, and cannot access each other's items.",
  "translation": "这台人工智能冰箱由五个室友共享，每个室友都标记自己的杂货。它跟踪物品、过期日期，并仅使用所有者的物品生成食谱。室友必须登录以管理他们的杂货，不能访问其他人的物品。"
}


In [29]:
try:
    json_response = json.loads(response)
    print(json_response)  # Valid JSON object
except json.JSONDecodeError as e:
    print(f"Error parsing JSON: {e}")

{'summary': "The AI refrigerator is shared by five housemates, each labeling their groceries. It tracks items, expiration dates, and generates recipes using only the owner's items. Housemates must log in to manage their groceries, and cannot access each other's items.", 'translation': '这台人工智能冰箱由五个室友共享，每个室友都标记自己的杂货。它跟踪物品、过期日期，并仅使用所有者的物品生成食谱。室友必须登录以管理他们的杂货，不能访问其他人的物品。'}


# 3. Chain of Thought

## 3.1 Example: without chain of thought

In [30]:
def chat_complete_prompt(text):
    # query against the model "gpt-3.5-turbo-1106"
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo-1106",
        messages= [{"role": "user", "content": text}],
        temperature=0, # this is the degree of randomness of the model's output
    )
    return completion.choices[0].message.content

In [31]:
text = f"""
Q: Take the last letters of each words in "Vinny Landon Miguel Caitlyn" and concatenate them.
"""
response = chat_complete_prompt(text)
print("The response is: ", response)

The response is:  A: The last letters of each word are "y n n l n n g n". Concatenating them gives "ynnlngn".


## 3.2 Example: with chain of thought

In [32]:
text = f"""
Q: Take the last letters of each words in "Vinny Landon Miguel Caitlyn" and concatenate them.
"""
prompt = f"""
Q: Take the last letters of each words in “Leonardo Piero da Vinci” and concatenate them.
A: Let's think step by step. The last letter of "Leonardo" is o, the last letter of "Piero" is o, 
the last letter of "da" is a, the the last letter of "Vinci" is i. So, the concatenated result is 'o'+'o'+'a'+'i', i.e. "ooai"
{text}


"""
response = chat_complete_prompt(prompt)
print("The response is: ", response)

The response is:  A: The last letter of "Vinny" is y, the last letter of "Landon" is n, the last letter of "Miguel" is l, and the last letter of "Caitlyn" is n. So, the concatenated result is 'y'+'n'+'l'+'n', i.e. "ynln"


# 4. Using Memory in LLM Calls

## 4.1 Environment Preparation

In [33]:
import os
from dotenv import load_dotenv

import openai
from openai import OpenAI

In [34]:
load_dotenv()

True

In [35]:
client = OpenAI()

## 4.2 Prepare a chatContext, with all the prompts and chat history

## Explanation: I modified the text with our project AI refrigerator. 

In [36]:
def chat_complete_messages(messages, temperature):
    # query against the model "gpt-3.5-turbo-1106"
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages= messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return completion.choices[0].message.content

In [37]:
chatContext = [
    {'role':'system', 'content': f"""
Objective: You are a smart, friendly virtual assistant tasked with managing refrigerator in my house. 

Procedure:
Begin with a greeting and offer assistance in adding new grocery items for my home. If a user fails to authenticate
him or her, you have to reject adding item and politely explain that user needs to provide a password. 
For inquiries beyond adding items, students can ask what AI agent can do it for him or herself. 

Task Steps:
1. Request the user's name, ask password. If it matches, goes to next step. 
2. If no password is provided, remind them that he or she can't use refrigerator without it. 
3. Display the current item list. If quantity is 0, drop it from the list. Then, continue with the adding grocery item process.
4. Confirm the user added the grocery items and how many they are. 
5. Inquire if the student wishes to add another grocery item. 
a. If yes, repeat step 3.
b. If no, proceed to step 6.

The final step:
6. Check if there are expiring grocery items in the previous check ins, if so 
ask whether the user throw them away or not.
a. If yes, remove the expiring items from the grocery list and let user know it. 
b, If no, proceed to step 7.
7.Finally, ask user whether it will generate a wonderful recipe for today with those grocery items. 
a. If yes, proceed 8.
8. Show the recipe and ask user to proceed it? 
a. If yes, deduct the quanity of items used in the recipe from the grocery item list.
b. If no, repeat step 3. 

[Default User:]
The user includes:
- Minsu, password:1234
[Default User's current grocery List:]
The grocery item includes:
- Egg, Quatity: 30, Expiration: November 10, 2024
- Tomatoes, Quantity: 5, Expiration: December 15, 2024 
- Broccoli, Quantity: 3, Expiration: January 1, 2025


"""},      
]
chatContext

[{'role': 'system',
  'content': "\nObjective: You are a smart, friendly virtual assistant tasked with managing refrigerator in my house. \n\nProcedure:\nBegin with a greeting and offer assistance in adding new grocery items for my home. If a user fails to authenticate\nhim or her, you have to reject adding item and politely explain that user needs to provide a password. \nFor inquiries beyond adding items, students can ask what AI agent can do it for him or herself. \n\nTask Steps:\n1. Request the user's name, ask password. If it matches, goes to next step. \n2. If no password is provided, remind them that he or she can't use refrigerator without it. \n3. Display the current item list. If quantity is 0, drop it from the list. Then, continue with the adding grocery item process.\n4. Confirm the user added the grocery items and how many they are. \n5. Inquire if the student wishes to add another grocery item. \na. If yes, repeat step 3.\nb. If no, proceed to step 6.\n\nThe final step:\n

<span style="color: green"> **The loop begins from here and ends with "exit"**</span>

In [38]:

while True:
    response_message_content = chat_complete_messages(chatContext, 0)
    print("ChatBot:", response_message_content)
        
    chatContext.append({'role': 'assistant', 'content': f"{response_message_content}"})
    
    user_input = input("User Input: ")
    if user_input.lower() == "exit":
        print("Exited chat")
        break
    
    chatContext.append({'role': 'user', 'content': user_input})


ChatBot: Hello! I'm here to help you manage your refrigerator. What's your name? And could you please provide your password?


User Input:  minsu 1234


ChatBot: Thank you, Minsu! Your password is correct. 

Here's your current grocery list:
- Egg, Quantity: 30, Expiration: November 10, 2024
- Tomatoes, Quantity: 5, Expiration: December 15, 2024 
- Broccoli, Quantity: 3, Expiration: January 1, 2025

What grocery item would you like to add? Please provide the item name and quantity.


User Input:  cabbage quantity:1 expiration:February 2025


ChatBot: Great! You've added 1 cabbage to your grocery list. 

Here's the updated grocery list:
- Egg, Quantity: 30, Expiration: November 10, 2024
- Tomatoes, Quantity: 5, Expiration: December 15, 2024 
- Broccoli, Quantity: 3, Expiration: January 1, 2025
- Cabbage, Quantity: 1, Expiration: February 2025

Would you like to add another grocery item?


...

ChatBot: Great! Here's a delicious recipe you can make with your grocery items:

**Vegetable Omelette**

**Ingredients:**
- 2 Eggs
- 1 Tomato, diced
- 1/2 Broccoli, chopped (you can use the remaining quantity)
- Salt and pepper to taste
- Optional: Cheese, herbs, or spices of your choice

**Instructions:**
1. In a bowl, beat the eggs and season with salt and pepper.
2. Heat a non-stick skillet over medium heat and add a little oil or butter.
3. Add the chopped broccoli and diced tomatoes to the skillet and sauté for a few minutes until they are tender.
4. Pour the beaten eggs over the vegetables in the skillet.
5. Cook until the edges start to set, then gently lift the edges with a spatula to allow uncooked egg to flow underneath.
6. Once the eggs are fully cooked, you can add cheese or herbs if desired.
7. Fold the omelette in half and serve hot!

Would you like to proceed with this recipe? If so, I will deduct the quantity of items used from your grocery list.


User Input:  yes


ChatBot: Perfect! I'll deduct the quantities used in the recipe:

- Eggs: 2 (from 30 to 28)
- Tomatoes: 1 (from 5 to 4)
- Broccoli: 0.5 (from 3 to 2.5)

Here's your updated grocery list:
- Egg, Quantity: 28, Expiration: November 10, 2024
- Tomatoes, Quantity: 4, Expiration: December 15, 2024 
- Broccoli, Quantity: 2.5, Expiration: January 1, 2025
- Cabbage, Quantity: 1, Expiration: February 2025

Enjoy your Vegetable Omelette! If you need anything else or want to add more grocery items, just let me know!


User Input:  exit


Exited chat
